In [34]:
import numpy as np
import torch
from dataLoader import *
from sklearn.decomposition import PCA

def resize_tensor(x,size = (7,7)):
    x = x.unsqueeze(0)
    x = torch.nn.functional.interpolate(x, size=size, mode='bilinear')
    return x.squeeze(0)

In [35]:
def load_images(dirName, reduce=1):
    files_train = sorted(dirName.glob("*.npz"))
    train_data = []

    for _, file_name in enumerate(files_train):
        with np.load(file_name) as npz:
            arr = np.ma.MaskedArray(**npz)
            data = arr.data
            data = np.append(data, [arr.mask[0]], axis=0)
            
            x = torch.tensor(data, dtype=torch.float32)
            x = resize_tensor(x)  # Shape: (C, 7, 7)
            train_data.append(x)

    if not train_data:
        return []

    max_width = max(data.shape[1] for data in train_data)
    max_height = max(data.shape[2] for data in train_data)
    max_size = max_width if max_width > max_height else max_height

    if reduce == 1:
        return [resize_tensor(data) for data in train_data]
        
    dataset_tensor = torch.stack(train_data)
    N, C, H, W = dataset_tensor.shape

    pixels = dataset_tensor.permute(0, 2, 3, 1).reshape(-1, C).numpy()

    pca = PCA(n_components=reduce)
    reduced_pixels = pca.fit_transform(pixels)

    actual_components = pca.n_components_

    reduced_tensor = torch.tensor(reduced_pixels, dtype=torch.float32)
    reduced_tensor = reduced_tensor.view(N, H, W, actual_components).permute(0, 3, 1, 2)

    return ([img for img in reduced_tensor], pca)

train_data, pca= load_images(HSI_SATELLITE_DIR, 0.95)

In [36]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

y = gt_train_df[column_names].values
y = scaler.fit_transform(y)
y_binned = np.digitize(y, bins=np.linspace(y.min(), y.max() + 1, 5))
y = torch.tensor(y, dtype=torch.float32)

# fig, axs = plt.subplots(ncols=6, nrows=3, figsize=(10,5))
# for i in range(6):
#     axs[0][i].hist(y[i])
#     axs[1][i].hist(y_train[i])
#     axs[2][i].hist(y_test[i])


# Prepare data and model to be used on the GPU

In [37]:
from torch.utils.data import DataLoader
from neural_networks.Convolutional import CNN2D
from sklearn.model_selection import train_test_split

train_dataset = list(zip(train_data, y))
train, test = train_test_split(train_dataset, train_size=0.8)

# MSI batch size: 64
# MSI neurons: 256
# HSI batch size: 128
# HSI neurons: 128
train_loader = DataLoader(train, batch_size=128, shuffle=True)
test_loader = DataLoader(test, batch_size=len(test), shuffle=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN2D(neurons=256, channels=3).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-5)
loss_fn = torch.nn.MSELoss()

best_val_loss = float('inf')
best_val_loss_before = best_val_loss

In [41]:
early_stopping_treshold = 128
early_stopping_counter = 0

for epoch in range(5000):
    model.train()
    train_loss= 0.0

    for x,y in train_loader:
        x = x.to(device)
        y = y.to(device)

        output = model(x)
        loss = loss_fn(output, y)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        train_loss+= loss.item()

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for x,y in test_loader:
            x = x.to(device)
            y = y.to(device)

            output = model(x)
            loss = loss_fn(output, y)

            val_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(test_loader)
    print(f"Epoch: {epoch}, train loss: {avg_train_loss:.4f} val loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        early_stopping_counter = 0

        torch.save(model.state_dict(), "models/nn/CNN2D_MSI_95.pt")
    else:
        early_stopping_counter += 1
        
        if early_stopping_counter >=early_stopping_treshold:
            break


# best score: 0.5883544087409973 for HSI without warm loading
# best score: 0.5423731803894043 for HSI with warm loading
# best score: 0.5265264511108398 for HSI kernel=1 with warm loading
# 0.5730111002922058 MSI kernel=3 with warm loading
print(best_val_loss)

Epoch: 0, train loss: 0.9914 val loss: 0.9418
Epoch: 1, train loss: 0.9837 val loss: 0.9320
Epoch: 2, train loss: 0.9751 val loss: 0.9170
Epoch: 3, train loss: 0.9618 val loss: 0.8959
Epoch: 4, train loss: 0.9403 val loss: 0.8698
Epoch: 5, train loss: 0.9207 val loss: 0.8467
Epoch: 6, train loss: 0.9039 val loss: 0.8297
Epoch: 7, train loss: 0.8931 val loss: 0.8187
Epoch: 8, train loss: 0.8831 val loss: 0.8104
Epoch: 9, train loss: 0.8817 val loss: 0.8036
Epoch: 10, train loss: 0.8731 val loss: 0.7970
Epoch: 11, train loss: 0.8692 val loss: 0.7926
Epoch: 12, train loss: 0.8605 val loss: 0.7868
Epoch: 13, train loss: 0.8556 val loss: 0.7823
Epoch: 14, train loss: 0.8495 val loss: 0.7773
Epoch: 15, train loss: 0.8469 val loss: 0.7766
Epoch: 16, train loss: 0.8438 val loss: 0.7708
Epoch: 17, train loss: 0.8390 val loss: 0.7695
Epoch: 18, train loss: 0.8419 val loss: 0.7668
Epoch: 19, train loss: 0.8358 val loss: 0.7659
Epoch: 20, train loss: 0.8338 val loss: 0.7627
Epoch: 21, train loss: 

# 3D convolutional neural network (Too slow for hsi)

In [45]:
from neural_networks.Convolutional import CNN3D

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN3D(neurons=256, channels=3).to(device)
optimiser = torch.optim.Adam(model.parameters(), lr=1e-4)
best_val_loss = float('inf')
best_val_loss_before = best_val_loss 
criterion = torch.nn.CrossEntropyLoss()
loss_fn = torch.nn.MSELoss()
early_stopping_treshold = 64

In [ ]:
def transform_images_with_pca(test_data, pca):
    dataset_tensor = torch.stack(test_data) if isinstance(test_data, list) else test_data

    N, C, H, W = dataset_tensor.shape
    pixels = dataset_tensor.permute(0, 2, 3, 1).reshape(-1, C).numpy()

    reduced_pixels = pca.transform(pixels)
    actual_components = pca.n_components_

    reduced_tensor = torch.tensor(reduced_pixels, dtype=torch.float32)
    return reduced_tensor.view(N, H, W, actual_components).permute(0, 3, 1, 2)

In [46]:
from pandas import DataFrame
from BaselineRegressor import BaselineRegressor
from sklearn.metrics import mean_squared_error

X_baseline = load_msi_data(HSI_SATELLITE_DIR)
y_baseline = gt_train_df[column_names].values
baseline = BaselineRegressor()
baseline.fit(X_baseline, y_baseline)
y_baseline_pred = baseline.predict(X_baseline)
mse_baseline = mean_squared_error(y_baseline, y_baseline_pred, multioutput='raw_values')

# model = CNN2D(channels=231, neurons=256, kernel_size=1)
model.load_state_dict(torch.load("models/nn/CNN3D_HSI_95.pt"))
model.to(device)
model.eval()

test_data = load_images(HSI_SATELLITE_TEST_DIR)
# test_data = torch.stack(test_data) # comment out for 2D Cnn
# train_data = torch.stack(train_data) # comment out for 3D Cnn
train_data = train_data.to(device)
with torch.no_grad():
    # predictions = model(test_data)
    predictions_score = model(train_data)

# predictions = predictions.cpu().numpy()
# predictions = scaler.inverse_transform(predictions)

predictions_score = predictions_score.cpu().numpy()
predictions_score = scaler.inverse_transform(predictions_score)
mse = mean_squared_error(y_baseline, predictions_score, multioutput='raw_values')

# submission = DataFrame(data = predictions, columns = column_names)
# submission.to_csv("cnn3dHsi_95.csv", index_label="sample_index")
score = mse / mse_baseline
score

array([0.49918575, 0.62049721, 0.45977483, 0.68627737, 0.61806255,
       0.53619069])

# Check models for

In [ ]:
from neural_networks.Convolutional import CNN2D
from neural_networks.Recurrent import FusionNN

def is_fusion_model(model):
    return hasattr(model, "encoders")
    
paths = [
    "models/nn/FusionNN.pt",
    "models/nn/CNN2D_HSI.pt",
    "models/nn/CNN2D_kernel1_MSI.pt"
]

models = [
    FusionNN(neurons=256, hidden_size=12000).to(device),
    CNN2D(neurons=256, channels=231).to(device),
    CNN2D(neurons=256, channels=231, kernel_size=1).to(device)
]

column_map = {
    0: 1, 
    1: 1, 
    2: 0,
    3: 1,
    4: 2,
    5: 0
}

predictions = [[],[],[]]

files_test_HSI = sorted(HSI_SATELLITE_TEST_DIR.glob("*.npz"))
files_test_MSI = sorted(MSI_SATELLITE_TEST_DIR.glob("*.npz"))

predictions = []

for i in range(len(models)):
    print(i)
    model = models[i]
    model.load_state_dict(torch.load(paths[i]))
    model.eval()

    model_preds = []

    with torch.no_grad():
        for idx in range(len(files_test_HSI)):

            # --- HSI ---
            with np.load(files_test_HSI[idx]) as npz:
                arr = np.ma.MaskedArray(**npz)
                data = arr.data
                data = np.append(data, [arr.mask[0]], axis=0)

                x_hsi = torch.tensor(data, dtype=torch.float32)
                x_hsi = resize_tensor(x_hsi)
                x_hsi = x_hsi.unsqueeze(0).to(device)

            # --- MSI ---
            with np.load(files_test_MSI[idx]) as npz:
                arr = np.ma.MaskedArray(**npz)
                data = arr.data
                data = np.append(data, [arr.mask[0]], axis=0)

                x_msi = torch.tensor(data, dtype=torch.float32)
                x_msi = resize_tensor(x_msi)
                x_msi = x_msi.unsqueeze(0).to(device)

            # --- MODEL FORWARD ---
            if hasattr(model, "encoders"):  # FusionNN
                output = model([x_msi, x_hsi])
            else:  # CNN2D
                output = model(x_hsi)

            model_preds.append(output.cpu())

    predictions.append(torch.cat(model_preds, dim=0))


In [ ]:
final_preds = torch.zeros_like(predictions[0])

for col in range(6):
    model_id = column_map[col]
    final_preds[:, col] = predictions[model_id][:, col]

final_preds = final_preds.cpu().numpy()
final_preds = scaler.inverse_transform(final_preds)

from pandas import DataFrame

submission = DataFrame(
    data=final_preds,
    columns=column_names
)

submission.to_csv("final.csv", index_label="sample_index")